In [1]:
from pyspark.sql import SparkSession

In [2]:
import psycopg2

In [3]:
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.eventLog.gcMetrics.youngGenerationGarbageCollectors", "G1 Young Generation") \
    .config("spark.eventLog.gcMetrics.oldGenerationGarbageCollectors", "G1 Old Generation") \
    .getOrCreate()

25/04/30 15:31:41 WARN Utils: Your hostname, milanthapa resolves to a loopback address: 127.0.1.1; using 10.10.42.122 instead (on interface enp2s0)
25/04/30 15:31:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/30 15:31:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/30 15:31:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
df = spark.read.parquet("/home/milan-thapa/Desktop/task 3/provider_output.parquet")
df1=spark.read.parquet("/home/milan-thapa/Desktop/task 3/net_output.parquet")

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="admin",
    host="localhost",
    port=5432
)

In [ ]:
jdbc_url = "jdbc:postgresql://localhost:5432/postgres"
connection_properties = {
    "user": "postgres",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

In [7]:
df1.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- billing_class: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- provided_group_id: long (nullable = true)
 |-- negotiated_type: string (nullable = true)



In [8]:
cur = conn.cursor()
create_table_query = """
DROP TABLE IF EXISTS in_network_data;
CREATE TABLE IF NOT EXISTS in_network_data (
    billing_code TEXT,
    billing_code_type TEXT,
    negotiation_arrangement TEXT,
    billing_code_modifier TEXT[],
    billing_class TEXT,
    negotiated_rate DOUBLE PRECISION,
    service_code INTEGER[],
    provider_group_id BIGINT,
    negotiated_type TEXT
);
"""

cur.execute(create_table_query)
conn.commit()

In [ ]:
df.write.jdbc(url=jdbc_url,table="in_network_data",mode="append", properties=connection_properties)
conn.commit()

AnalysisException: Column provided_group_id not found in schema Some(StructType(StructField(billing_code,StringType,true),StructField(billing_code_type,StringType,true),StructField(negotiation_arrangement,StringType,true),StructField(billing_code_modifier,ArrayType(StringType,true),true),StructField(billing_class,StringType,true),StructField(negotiated_rate,DoubleType,true),StructField(service_code,ArrayType(IntegerType,true),true),StructField(provider_group_id,LongType,true),StructField(negotiated_type,StringType,true))).

In [ ]:
cur.close()
conn.close()

In [ ]:
# dfa=spark.read.parquet("provider_output.parquet")
# dfa.show(20)